# Optimization Algorithms for Machine Learning
Implementing and visualizing optimizers from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. The Rosenbrock Function

In [ ]:
def rosenbrock(x, y):
    return (1 - x)**2 + 100*(y - x**2)**2

def rosenbrock_grad(x, y):
    dx = -2*(1-x) - 400*x*(y - x**2)
    dy = 200*(y - x**2)
    return np.array([dx, dy])

x = np.linspace(-2, 2, 300)
y = np.linspace(-1, 3, 300)
X, Y = np.meshgrid(x, y)
Z = rosenbrock(X, Y)

plt.figure(figsize=(10, 7))
plt.contour(X, Y, Z, levels=np.logspace(-1, 3.5, 30), norm=LogNorm(), cmap='viridis')
plt.colorbar(label='f(x,y)')
plt.scatter([1], [1], color='red', s=200, marker='*', label='Minimum (1,1)')
plt.title('Rosenbrock Function Contours')
plt.xlabel('x'); plt.ylabel('y')
plt.legend()
plt.show()

## 2. Implementing Optimizers from Scratch

In [ ]:
class SGD:
    def __init__(self, lr=0.001):
        self.lr = lr
    def step(self, params, grads):
        return params - self.lr * grads

class Momentum:
    def __init__(self, lr=0.001, beta=0.9):
        self.lr, self.beta = lr, beta
        self.v = None
    def step(self, params, grads):
        if self.v is None:
            self.v = np.zeros_like(params)
        self.v = self.beta * self.v + grads
        return params - self.lr * self.v

class RMSProp:
    def __init__(self, lr=0.001, beta=0.999, eps=1e-8):
        self.lr, self.beta, self.eps = lr, beta, eps
        self.s = None
    def step(self, params, grads):
        if self.s is None:
            self.s = np.zeros_like(params)
        self.s = self.beta * self.s + (1 - self.beta) * grads**2
        return params - self.lr * grads / (np.sqrt(self.s) + self.eps)

class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self.m = self.v = None
        self.t = 0
    def step(self, params, grads):
        if self.m is None:
            self.m = np.zeros_like(params)
            self.v = np.zeros_like(params)
        self.t += 1
        self.m = self.beta1*self.m + (1-self.beta1)*grads
        self.v = self.beta2*self.v + (1-self.beta2)*grads**2
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)
        return params - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

print("All optimizers implemented!")

## 3. Comparing Optimizers on Rosenbrock

In [ ]:
def optimize(optimizer, grad_fn, start, n_steps=5000):
    path = [start.copy()]
    params = start.copy()
    for _ in range(n_steps):
        g = grad_fn(params[0], params[1])
        params = optimizer.step(params, g)
        path.append(params.copy())
    return np.array(path)

start = np.array([-1.5, 1.5])
paths = {
    'SGD (lr=0.0005)': optimize(SGD(lr=0.0005), rosenbrock_grad, start),
    'Momentum (lr=0.0002)': optimize(Momentum(lr=0.0002, beta=0.9), rosenbrock_grad, start),
    'RMSProp (lr=0.002)': optimize(RMSProp(lr=0.002), rosenbrock_grad, start),
    'Adam (lr=0.005)': optimize(Adam(lr=0.005), rosenbrock_grad, start),
}

plt.figure(figsize=(12, 8))
plt.contour(X, Y, Z, levels=np.logspace(-1, 3.5, 20), norm=LogNorm(), cmap='gray', alpha=0.5)
colors = ['blue', 'red', 'green', 'orange']
for (name, path), c in zip(paths.items(), colors):
    plt.plot(path[:500, 0], path[:500, 1], '-', color=c, lw=1.5, label=name, alpha=0.8)
    plt.scatter(path[0, 0], path[0, 1], color=c, s=50, marker='o')
plt.scatter([1], [1], color='gold', s=300, marker='*', zorder=5, label='Minimum')
plt.legend(fontsize=10)
plt.title('Optimizer Comparison on Rosenbrock Function')
plt.xlabel('x'); plt.ylabel('y')
plt.xlim(-2, 2); plt.ylim(-1, 3)
plt.show()

## 4. Learning Rate Effect

In [ ]:
# Simple quadratic: f(x) = x^2
def simple_gd(lr, start=5.0, steps=20):
    path = [start]
    x = start
    for _ in range(steps):
        x = x - lr * 2 * x  # gradient of x^2 is 2x
        path.append(x)
    return path

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
lrs = [0.01, 0.45, 1.05]
titles = ['Too Small (lr=0.01)', 'Just Right (lr=0.45)', 'Too Large (lr=1.05)']

x_plot = np.linspace(-6, 6, 100)
for ax, lr, title in zip(axes, lrs, titles):
    path = simple_gd(lr)
    ax.plot(x_plot, x_plot**2, 'b-', lw=2)
    for i in range(min(len(path)-1, 15)):
        ax.annotate('', xy=(path[i+1], path[i+1]**2), xytext=(path[i], path[i]**2),
                   arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
    ax.scatter(path[:15], [p**2 for p in path[:15]], color='red', s=30, zorder=5)
    ax.set_title(title)
    ax.set_ylim(-1, 30)
plt.tight_layout()
plt.show()

## 5. Convex vs Non-Convex Functions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Convex
x = np.linspace(-3, 3, 300)
axes[0].plot(x, x**2, 'b-', lw=2)
axes[0].scatter([0], [0], color='red', s=100, zorder=5)
axes[0].set_title('Convex: f(x) = x² \n(Single global minimum)')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.3)

# Non-convex
y_nc = x**4 - 5*x**2 + 4 + 0.5*x
axes[1].plot(x, y_nc, 'b-', lw=2)
# Find local minima numerically
from scipy.signal import argrelmin
mins = argrelmin(y_nc, order=10)[0]
axes[1].scatter(x[mins], y_nc[mins], color='red', s=100, zorder=5)
axes[1].set_title('Non-Convex: f(x) = x⁴ - 5x² + 4 + 0.5x\n(Multiple local minima)')
axes[1].axhline(y=min(y_nc[mins]), color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate getting stuck in local minimum
def non_convex(x):
    return x**4 - 5*x**2 + 4 + 0.5*x

def non_convex_grad(x):
    return 4*x**3 - 10*x + 0.5

starts = [-2.5, -0.5, 0.5, 2.5]
plt.figure(figsize=(10, 5))
x_plot = np.linspace(-3, 3, 300)
plt.plot(x_plot, non_convex(x_plot), 'k-', lw=2)

for s in starts:
    path = [s]
    x = s
    for _ in range(100):
        x = x - 0.01 * non_convex_grad(x)
        path.append(x)
    path = np.array(path)
    plt.plot(path, non_convex(path), '.-', markersize=3, alpha=0.7, label=f'Start={s}')
    plt.scatter([path[-1]], [non_convex(path[-1])], s=100, zorder=5)

plt.legend()
plt.title('Different Starting Points → Different Minima')
plt.show()

## 6. Learning Rate Schedules

In [ ]:
epochs = np.arange(100)

def step_schedule(epoch, lr0=0.1, drop=0.5, every=30):
    return lr0 * drop**(epoch // every)

def cosine_schedule(epoch, lr0=0.1, T=100):
    return lr0 * 0.5 * (1 + np.cos(np.pi * epoch / T))

def warmup_cosine(epoch, lr0=0.1, warmup=10, T=100):
    if epoch < warmup:
        return lr0 * epoch / warmup
    return lr0 * 0.5 * (1 + np.cos(np.pi * (epoch - warmup) / (T - warmup)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, [step_schedule(e) for e in epochs], 'b-', lw=2, label='Step Decay')
ax.plot(epochs, [cosine_schedule(e) for e in epochs], 'r-', lw=2, label='Cosine Annealing')
ax.plot(epochs, [warmup_cosine(e) for e in epochs], 'g-', lw=2, label='Warmup + Cosine')
ax.set_xlabel('Epoch'); ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedules')
ax.legend()
plt.show()

## 7. Convergence Speed Comparison

In [ ]:
# Compare convergence on a simple quadratic
def quadratic(params):
    return 0.5 * (params[0]**2 + 10*params[1]**2)

def quadratic_grad(params):
    return np.array([params[0], 10*params[1]])

start = np.array([5.0, 3.0])
n_steps = 200

optimizers = {
    'SGD': SGD(lr=0.05),
    'Momentum': Momentum(lr=0.05, beta=0.9),
    'RMSProp': RMSProp(lr=0.1),
    'Adam': Adam(lr=0.1),
}

plt.figure(figsize=(10, 5))
for name, opt in optimizers.items():
    params = start.copy()
    losses = []
    if hasattr(opt, 'v'): opt.v = None
    if hasattr(opt, 's'): opt.s = None
    if hasattr(opt, 'm'): opt.m = None; opt.t = 0
    for _ in range(n_steps):
        losses.append(quadratic(params))
        g = quadratic_grad(params)
        params = opt.step(params, g)
    plt.semilogy(losses, lw=2, label=name)

plt.xlabel('Iteration'); plt.ylabel('Loss (log scale)')
plt.title('Convergence Speed Comparison')
plt.legend()
plt.show()

## 8. Gradient Descent with Noise (Simulating SGD)

In [ ]:
# Show how noise in SGD helps escape local minima
def bumpy_loss(x):
    return x**2 + 3*np.sin(3*x)

def bumpy_grad(x):
    return 2*x + 9*np.cos(3*x)

x_plot = np.linspace(-5, 5, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pure GD - gets stuck
for start in [-4, -2, 0.5, 3]:
    x = start
    path = [x]
    for _ in range(100):
        x = x - 0.01 * bumpy_grad(x)
        path.append(x)
    axes[0].plot(path, bumpy_loss(np.array(path)), '.-', markersize=3, alpha=0.7)
axes[0].plot(x_plot, bumpy_loss(x_plot), 'k-', lw=2, alpha=0.3)
axes[0].set_title('Pure GD (gets stuck)')

# Noisy GD (SGD-like) - can escape
np.random.seed(42)
for start in [-4, -2, 0.5, 3]:
    x = start
    path = [x]
    for _ in range(200):
        noise = np.random.randn() * 2
        x = x - 0.01 * (bumpy_grad(x) + noise)
        path.append(x)
    axes[1].plot(path, bumpy_loss(np.array(path)), '.-', markersize=2, alpha=0.7)
axes[1].plot(x_plot, bumpy_loss(x_plot), 'k-', lw=2, alpha=0.3)
axes[1].set_title('Noisy GD / SGD (escapes local minima)')
plt.tight_layout()
plt.show()

## 9. Optimizer Hyperparameter Sensitivity

In [ ]:
# Adam beta1 sensitivity
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
start = np.array([-1.5, 1.5])

for beta1 in [0.5, 0.8, 0.9, 0.99]:
    opt = Adam(lr=0.005, beta1=beta1)
    path = optimize(opt, rosenbrock_grad, start, n_steps=3000)
    losses = [rosenbrock(p[0], p[1]) for p in path]
    axes[0].semilogy(losses[:500], label=f'β1={beta1}')
axes[0].set_title('Adam: Effect of β1'); axes[0].legend(); axes[0].set_ylabel('Loss')

for lr in [0.001, 0.005, 0.01, 0.05]:
    opt = Adam(lr=lr)
    path = optimize(opt, rosenbrock_grad, start, n_steps=3000)
    losses = [rosenbrock(p[0], p[1]) for p in path]
    axes[1].semilogy(losses[:500], label=f'lr={lr}')
axes[1].set_title('Adam: Effect of Learning Rate'); axes[1].legend()
plt.tight_layout()
plt.show()

## Summary

| Optimizer | Key Idea | Pros | Cons |
|-----------|----------|------|------|
| SGD | Raw gradient | Simple | Slow, sensitive to LR |
| Momentum | Accumulate velocity | Faster convergence | Extra hyperparameter |
| RMSProp | Adaptive per-param LR | Good for sparse | Can be unstable |
| Adam | Momentum + RMSProp | Best general-purpose | Memory overhead |